In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/cheminfo-models-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "03-04.py"

## MoleculeNet 與 DeepChem：機器學習用的標準資料集

首格準備執行環境，下一格寫出分析程式，末格執行並顯圖。練習可接在分析程式末端。第一次執行需連線取得 MoleculeNet 資料。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile
import deepchem as dc
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path(tempfile.gettempdir()) / "book-moleculenet"))
cache.mkdir(parents=True, exist_ok=True)
options = dict(featurizer=dc.feat.MACCSKeysFingerprint(), splitter="scaffold",
               transformers=[], data_dir=str(cache), save_dir=str(cache), reload=False)
folder = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
folder.mkdir(exist_ok=True)

tasks, parts, transforms = dc.molnet.load_delaney(**options)
train, valid, test = parts
for name, data in zip(("train", "valid", "test"), parts):
    assert len(data.ids) == data.X.shape[0] == data.y.shape[0]
    assert data.w.shape == data.y.shape
    print(name, data.X.shape, data.y.shape, np.count_nonzero(data.w[:, 0]))
print("task:", tasks[0], "transforms:", len(transforms))
print("first training structure:", train.ids[0], "logS:", train.y[0, 0])
assert np.all((train.X == 0) | (train.X == 1))
# 三幅圖採相同的橫軸與分箱，也保留各組的實際筆數。
fig, axes = plt.subplots(3, 1, sharex=True, figsize=(7.2, 5.4), layout="constrained")
edges = np.linspace(-13, 3, 25)
for ax, name, data in zip(axes, ("Train", "Validation", "Test"), parts):
    values = data.y[data.w[:, 0] > 0, 0]
    assert values.min() >= edges[0] and values.max() <= edges[-1]
    ax.hist(values, bins=edges, color="#237283")
    ax.set(ylabel="Count", title=f"{name} (n={len(values)})")
axes[-1].set_xlabel("Measured logS (S in mol/L)")
fig.savefig(folder / "03-04-esol-splits.png", dpi=150)

In [ ]:
from IPython.display import Image, display
subprocess.run([python, str(script)], check=True)
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
display(Image(filename=str(out / "03-04-esol-splits.png")))